# 面试问题：LLM Prefix Cache 怎样设计？Radix Tree、块复用、失效、租户隔离与调度如何实现？

**一句话回答**：以完整 token 序列而非原始字符串匹配最长前缀，把可复用的 KV block 挂在前缀树上；cache identity 必须绑定模型权重、adapter、tokenizer/chat template、位置编码和权限域。命中只复用完整且已提交的 blocks，尾部 Copy-on-Write，并用引用计数与 LRU/成本策略回收。

本 Notebook 手写 cache key、Radix 风格 token trie、块对齐命中、引用计数、租户隔离、版本失效、reuse-aware 调度和 TTFT 收益核算。


In [ ]:
from dataclasses import dataclass,field  # 导入本单元所需的依赖。
from collections import OrderedDict  # 导入本单元所需的依赖。
import hashlib,math  # 导入本单元所需的依赖。

SEED132=13201; BLOCK132=4  # 计算并保存当前步骤的中间状态。
assert SEED132==13201  # 用受控断言验证关键不变量。
assert BLOCK132>0 and BLOCK132&(BLOCK132-1)==0  # 用受控断言验证关键不变量。
assert hashlib.sha256(b"prefix").hexdigest()!=hashlib.sha256(b"Prefix").hexdigest()  # 用受控断言验证关键不变量。


## 1. Cache identity 比 prompt 文本更严格

两段肉眼相同的文本可能因 tokenizer、special token、chat template 或 adapter 不同而生成不同 KV；同一 token 序列在不同模型 revision 下也绝不可混用。权限域必须进入 key，避免共享系统提示或检索证据造成跨租户侧信道。


In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class CacheScope132:  # 定义承载本节状态与行为的数据结构。
    model_rev:str; tokenizer_rev:str; template_rev:str; adapter_rev:str; tenant:str; rope_rev:str="base"  # 计算并保存当前步骤的中间状态。
    def digest(self): return hashlib.sha256(repr(self).encode()).hexdigest()  # 定义本节可复用的核心函数。
scope_a132=CacheScope132("m1","t1","c1","none","A"); scope_b132=CacheScope132("m1","t1","c1","none","B")  # 计算并保存当前步骤的中间状态。
assert scope_a132.digest()!=scope_b132.digest()  # 用受控断言验证关键不变量。
assert len(scope_a132.digest())==64  # 用受控断言验证关键不变量。
assert scope_a132.model_rev=="m1"  # 用受控断言验证关键不变量。


## 2. 前缀树按 token ID 做 longest-prefix match

字符串前缀不等于 token 前缀；缓存查找必须发生在最终模板渲染和 tokenization 之后。教学实现每条边一个 token，生产 Radix Tree 会压缩连续边以减少节点。节点只引用已完成 KV blocks，不保存未提交的半成品。


In [ ]:
@dataclass  # 为下方定义附加声明式配置。
class Node132:  # 定义承载本节状态与行为的数据结构。
    children:dict=field(default_factory=dict); blocks:tuple=(); committed:bool=False  # 计算并保存当前步骤的中间状态。
class Trie132:  # 定义承载本节状态与行为的数据结构。
    def __init__(self): self.root=Node132()  # 定义本节可复用的核心函数。
    def insert(self,tokens,blocks):  # 定义本节可复用的核心函数。
        n=self.root  # 计算并保存当前步骤的中间状态。
        for t in tokens: n=n.children.setdefault(t,Node132())  # 遍历输入元素以累积或检查结果。
        n.blocks=tuple(blocks); n.committed=True  # 计算并保存当前步骤的中间状态。
    def longest(self,tokens):  # 定义本节可复用的核心函数。
        n=self.root; best=(0,())  # 计算并保存当前步骤的中间状态。
        for i,t in enumerate(tokens,1):  # 遍历输入元素以累积或检查结果。
            if t not in n.children: break  # 按当前条件选择后续控制路径。
            n=n.children[t]  # 计算并保存当前步骤的中间状态。
            if n.committed: best=(i,n.blocks)  # 按当前条件选择后续控制路径。
        return best  # 返回当前分支计算出的结果。
trie132=Trie132(); trie132.insert((1,2,3,4),(10,)); trie132.insert((1,2,3,4,5,6,7,8),(10,11))  # 计算并保存当前步骤的中间状态。
assert trie132.longest((1,2,3,4,9))==(4,(10,))  # 用受控断言验证关键不变量。
assert trie132.longest((1,2,3,4,5,6,7,8,9))[0]==8  # 用受控断言验证关键不变量。
assert trie132.longest((9,))[0]==0  # 用受控断言验证关键不变量。


## 3. 只复用位置一致的完整 block

若 block size 为 4，命中 6 个 token 通常只安全复用前 4 个，剩余 2 个重新 prefill；共享不完整尾块会让后续 append 覆盖别人的 KV。某些 schema cache 支持模块化位置映射，但必须证明 RoPE position 与 attention mask 一致。


In [ ]:
def reusable132(matched_tokens,block_size): return matched_tokens//block_size*block_size  # 定义本节可复用的核心函数。
assert reusable132(6,4)==4  # 用受控断言验证关键不变量。
assert reusable132(8,4)==8  # 用受控断言验证关键不变量。
assert reusable132(3,4)==0  # 用受控断言验证关键不变量。


## 4. 缓存所有权与请求引用分开

Cache entry 持有一份引用，活跃请求命中后再 retain；LRU 淘汰只能释放 cache 引用，不能回收仍被请求使用的 block。写路径产生新尾块，完成后才原子发布到 trie，失败请求不得污染公共缓存。


In [ ]:
class Refs132:  # 定义承载本节状态与行为的数据结构。
    def __init__(self): self.r={}  # 定义本节可复用的核心函数。
    def retain(self,b): self.r[b]=self.r.get(b,0)+1  # 定义本节可复用的核心函数。
    def release(self,b):  # 定义本节可复用的核心函数。
        self.r[b]-=1  # 计算并保存当前步骤的中间状态。
        if self.r[b]==0: del self.r[b]  # 按当前条件选择后续控制路径。
refs132=Refs132(); [refs132.retain(b) for b in (10,11)]; refs132.retain(10); refs132.release(10)  # 计算并保存当前步骤的中间状态。
assert refs132.r[10]==1  # 用受控断言验证关键不变量。
assert refs132.r[11]==1  # 用受控断言验证关键不变量。
refs132.release(11); assert 11 not in refs132.r  # 执行当前语句以推进本节示例。


## 5. 淘汰目标是释放 block，而非删除最多 key

前缀节点共享祖先 blocks，删除一个叶子未必释放任何物理内存。策略可结合 `last_access`、独占 block 数、重算成本和租户配额。下面用简化 LRU 在 entry 层选择，但显式返回实际可释放的独占 blocks。


In [ ]:
entries132=OrderedDict([("a",(1,2)),("b",(1,3)),("c",(4,5))]); active_refs132={1:2,2:1,3:1,4:1,5:1}  # 计算并保存当前步骤的中间状态。
def evict_one132(entries,refs):  # 定义本节可复用的核心函数。
    key,blocks=entries.popitem(last=False); freed=[]  # 计算并保存当前步骤的中间状态。
    for b in blocks:  # 遍历输入元素以累积或检查结果。
        refs[b]-=1  # 计算并保存当前步骤的中间状态。
        if refs[b]==0: freed.append(b); del refs[b]  # 按当前条件选择后续控制路径。
    return key,freed  # 返回当前分支计算出的结果。
key132,freed132=evict_one132(entries132,active_refs132)  # 计算并保存当前步骤的中间状态。
assert key132=="a"  # 用受控断言验证关键不变量。
assert freed132==[2]  # 用受控断言验证关键不变量。
assert active_refs132[1]==1  # 用受控断言验证关键不变量。


## 6. 跨租户共享默认关闭

即使 KV 不直接可读，命中带来的 TTFT 差异也可能泄露某个前缀是否存在。敏感系统应按 tenant/ACL 分区，并对日志隐藏原始 token；若允许公共前缀共享，只共享经过审核、不可包含用户数据的静态模块。


In [ ]:
caches132={}  # 计算并保存当前步骤的中间状态。
def namespace132(scope): return (scope.tenant,scope.model_rev,scope.adapter_rev,scope.template_rev)  # 定义本节可复用的核心函数。
caches132.setdefault(namespace132(scope_a132),Trie132()).insert((1,2,3,4),(7,))  # 执行当前语句以推进本节示例。
assert namespace132(scope_a132)!=namespace132(scope_b132)  # 用受控断言验证关键不变量。
assert namespace132(scope_b132) not in caches132  # 用受控断言验证关键不变量。
assert caches132[namespace132(scope_a132)].longest((1,2,3,4))[0]==4  # 用受控断言验证关键不变量。


## 7. 版本切换采用 namespace replacement

新权重、LoRA、量化、RoPE scaling、tokenizer 或 template 任一变化都创建新 namespace；旧缓存自然 drain 后删除，避免扫描修改每个 entry。部署 rollback 则切回对应 immutable namespace，而不是复用“看起来一样”的 KV。


In [ ]:
old132=CacheScope132("m1","t1","c1","none","A"); new132=CacheScope132("m2","t1","c1","none","A")  # 计算并保存当前步骤的中间状态。
namespaces132={old132.digest():{"state":"draining"},new132.digest():{"state":"active"}}  # 计算并保存当前步骤的中间状态。
assert old132.digest()!=new132.digest()  # 用受控断言验证关键不变量。
assert namespaces132[new132.digest()]["state"]=="active"  # 用受控断言验证关键不变量。
assert namespaces132[old132.digest()]["state"]=="draining"  # 用受控断言验证关键不变量。


## 8. Reuse-aware 调度不能饿死冷前缀

把命中 token 转成预计 prefill 节省，再减等待年龄惩罚/加公平优先级；只按最长命中排序会让新租户和独特请求长期饥饿。线上同时看 cache hit token ratio、TTFT、eviction churn、隔离拒绝和不同租户 p95。


In [ ]:
requests132=[{"id":"hot","hit":800,"age":1},{"id":"cold","hit":0,"age":20},{"id":"mid","hit":300,"age":5}]  # 计算并保存当前步骤的中间状态。
def priority132(r): return r["hit"]*.01+r["age"]*.5  # 定义本节可复用的核心函数。
order132=sorted(requests132,key=priority132,reverse=True)  # 计算并保存当前步骤的中间状态。
assert order132[0]["id"]=="cold"  # 用受控断言验证关键不变量。
assert priority132(requests132[0])==8.5  # 用受控断言验证关键不变量。
assert {r["id"] for r in order132}=={"hot","cold","mid"}  # 用受控断言验证关键不变量。


## 面试总结

设计主线是：**最终 token IDs 做匹配 → identity 绑定 model/tokenizer/template/adapter/RoPE/tenant → trie 最长前缀 → 只复用完整已提交 blocks → cache/request 双引用 → 尾块 COW → 淘汰计算实际独占页 → namespace 版本失效 → reuse 与等待年龄联合调度 → 用 hit-token ratio 和 TTFT 验证**。字符串相同、模型名相同都不足以证明 KV 可复用。

延伸阅读：[Prompt Cache](https://arxiv.org/abs/2311.04934)、[SGLang / RadixAttention](https://arxiv.org/abs/2312.07104)、[PagedAttention](https://arxiv.org/abs/2309.06180)。
